In [1]:

from umap import UMAP
from hdbscan import HDBSCAN
from bertopic import BERTopic
from nltk.corpus import stopwords
from bertopic.representation import MaximalMarginalRelevance, PartOfSpeech, LangChain, KeyBERTInspired
from bertopic.vectorizers import  ClassTfidfTransformer
from sklearn.feature_extraction.text import CountVectorizer
import pandas as pd
from collections import Counter
import plotly.io as pio
import re
pio.renderers.default = "vscode"
stoplist = list(set(stopwords.words('english')))

In [2]:
thesis_df = pd.read_csv("themes_from_thesis.csv", sep="\t")
themes = thesis_df["Themes"]
all_themes = []
for x in themes:
    all_themes.extend(str(x).split("# "))
Counter(all_themes)

Counter({'sirens': 15,
         'port-dues': 11,
         'shallows': 11,
         'storm': 9,
         'assault-probability': 8,
         'birds': 7,
         'land & sea breezes': 6,
         'nocturnal & diurnal winds': 5,
         'indicators of coastal proximity': 5,
         'nan': 4,
         'unsafe anchorage': 4,
         'weather forecast': 4,
         'assault probability': 4,
         'harbourless shore': 3,
         'shoals': 3,
         'unsafe shore': 3,
         'storms': 3,
         'sea-monsters': 3,
         'pirates': 3,
         'shelters': 3,
         'hostile shores': 3,
         'bad anchorage': 2,
         'anchorage': 2,
         'shipping control': 2,
         'hazardous landing': 2,
         'contrary winds': 2,
         'swell': 2,
         'untrustworthy breezes': 2,
         'adverse winds': 2,
         'breeze': 2,
         'seamanship': 2,
         'shore': 2,
         'safe anchorages': 2,
         'unfriendly shores': 2,
         'lack of visibility':

In [3]:
seed_words = []
for x in themes:
    try:
        words = [re.sub(r"\s+", "-", i.lstrip().rstrip().replace("&", "").replace(":", "").replace("(", "").replace(")", "").replace("/","-")) for i in x.split("#") if i!=""]
        if len(words)>1:
            seed_words.append([re.sub("-+", "-",i) for i in words])
    except:
        continue

In [4]:
seed_words = list(set(map(lambda i: tuple(sorted(i)), seed_words)))

In [5]:
seed_words = [list(x) for x in seed_words]
seed_words

[['protection-asylum', 'sanctuary'],
 ['breeze-smell', 'indicators-of-coastal-proximity'],
 ['familiarity-with-the-place', 'harbour-access', 'shelter-during-storms'],
 ['egypt', 'revenues', 'tolls'],
 ['alexandria’s-port',
  'environmental-hazards',
  'geomorphology-shallows',
  'harbours-availability',
  'harbours-entrance',
  'pharos',
  'port-capacity-depth'],
 ['bad-anchorage', 'shoals'],
 ['anchorage', 'bottom-nature'],
 ['fake-torches', 'harbourless-shore', 'human-hazard', 'shallows'],
 ['political-factors', 'ports', 'treaties'],
 ['birds', 'coastal-proximity'],
 ['assault-probability', 'nocturnal-navigation', 'visibility'],
 ['risky-land-proximity', 'shallows', 'tide', 'unknown-shore'],
 ['attack-probability', 'unsafe-shore'],
 ['assault-probability', 'unfriendly-shores'],
 ['assault-probability', 'shallows'],
 ['geomorphological-hazard', 'harbourless-shore'],
 ['breakers', 'indicators-of-coastal-proximity', 'invisible-land', 'mist'],
 ['shallows', 'vessel-type-limitations'],
 [

In [6]:
id2label = {0: 'HIGH', 1: 'MEDIUM', 2: 'LOW'}

In [7]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2", model_kwargs={"torch_dtype": "float16"})

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [8]:
thesis_df.head()

,EVENT DATE,"AUTHOR, TEXTS",Abbreviations,TEXT DATE,EXCERPT,SOURCE LINK,Themes
0,ca. 330 BCE,"Aeschines, Against Ctesiphon, 112",Aeschin. In Ctes,ca. 330 BCE,"[...] ""Oaths"" ""Curse"" This curse, these oaths,...",https://topostext.org/work/106#112,port-dues
1,ca. 330 BCE,"Aeschines, Against Ctesiphon, 119",Aeschin. In Ctes,ca. 330 BCE,"You know of your own knowledge, and have no ne...",https://topostext.org/work/106#119,port-dues
2,Myth.,"Aeschylus, Suppliant Maidens, 755",Aesch. Supp.,ca. 463 BCE,DANAUS: A fleet in getting under way is not so...,https://topostext.org/work/11#755,landing risks# harbourless coast
3,ca. 360 CE,"Ammianus Marcellinus, History, 19.10.4",Amm. Marc.,ca. 390 CE,§ 19.10.4 And presently by the will of the di...,https://topostext.org/work/493#19.10.4,land & sea breezes# nocturnal & diurnal winds
4,ca. 360 CE,"Ammianus Marcellinus, History, 20.1.3",Amm. Marc.,ca. 390 CE,"§ 20.1.3 Therefore, taking the light-armed au...",https://topostext.org/work/493#20.1.3,land & sea breezes# nocturnal & diurnal winds


In [9]:
df = pd.read_csv("val_data.csv")

In [10]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    tokens = text.split()

    tokens = [t for t in tokens if t not in stoplist]
    return " ".join(tokens)

df["clean_text"] = df["text"].apply(clean_text)
thesis_df["clean_text"] = thesis_df["EXCERPT"].apply(clean_text)

In [11]:
texts = thesis_df["clean_text"]

In [12]:
embeddings = embedding_model.encode(list(texts), show_progress_bar=True)

Batches:   0%|          | 0/6 [00:00<?, ?it/s]

In [13]:
params = {  
            # TFIDF
            "reduce_frequent_words": True, "bm25_weighting": False,
            "seed_words": [],
            "seed_multiplier": 4,
            # UMAP
            "n_neighbors": 10, "n_components": 5, "min_dist": 0.0, "metric_umap": "cosine", "random_state": 42,
            # HDBSCAN (change min_cluster_size for more/less topics?, default is 10, recommended to only increase above 10)
            "min_cluster_size":2, "metric_hbd": "euclidean", "cluster_selection_method": "eom", "prediction_data": True,
            # Vectorizer model
            "stop_words": "english", "min_df": 1, "ngram_range": (1,4),
            # Representation models
            "diversity": 0.4
         }

In [14]:
ctfidf_model = ClassTfidfTransformer(reduce_frequent_words=params["reduce_frequent_words"], bm25_weighting=params["bm25_weighting"],
                                     seed_words=params['seed_words'], seed_multiplier=params["seed_multiplier"])



In [15]:

umap_model = UMAP(n_neighbors=params["n_neighbors"], 
                  n_components=params["n_components"], 
                  min_dist=params["min_dist"], 
                  metric=params["metric_umap"], 
                  random_state=params["random_state"])



In [16]:
hdbscan_model = HDBSCAN(min_cluster_size=params["min_cluster_size"],
                        metric=params["metric_hbd"], 
                        cluster_selection_method=params["cluster_selection_method"], 
                        prediction_data=params["prediction_data"])

In [17]:
vectorizer_model = CountVectorizer(stop_words=params["stop_words"], 
                                   min_df=params["min_df"], 
                                   ngram_range=params["ngram_range"])



In [18]:

representation_models = [
                            MaximalMarginalRelevance(diversity=params["diversity"]),
                             KeyBERTInspired(),
                            # PartOfSpeech("en_core_web_sm"),
                            ]

In [19]:
topic_model = BERTopic(

    # Pipeline models
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    # representation_model=representation_models,
    top_n_words=10,
    verbose=True,
    ctfidf_model=ctfidf_model,
    # nr_topics="auto",
    calculate_probabilities=True,
    seed_topic_list=seed_words,
)

# Train model
topics, probs = topic_model.fit_transform(texts, embeddings)
# new_topics = topic_model.reduce_outliers(texts, topics, probabilities=probs, threshold=0.03, strategy="probabilities")
# topic_model.update_topics(texts, topics=new_topics)


2026-05-07 16:49:43,940 - BERTopic - Guided - Find embeddings highly related to seeded topics.


Batches:   0%|          | 0/3 [00:00<?, ?it/s]

2026-05-07 16:49:43,980 - BERTopic - Guided - Completed ✓
2026-05-07 16:49:43,980 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-05-07 16:49:49,690 - BERTopic - Dimensionality - Completed ✓
2026-05-07 16:49:49,692 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-05-07 16:49:49,702 - BERTopic - Cluster - Completed ✓
2026-05-07 16:49:49,705 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-05-07 16:49:50,177 - BERTopic - Representation - Completed ✓


In [20]:
# id2label = {0: 'HIGH', 1: 'MEDIUM', 2: 'LOW'}
# labels = [id2label[x] for x in list(df["label"])]

In [21]:
# topics_per_class = topic_model.topics_per_class(texts, classes=labels)

In [22]:
27/len(texts)

0.15976331360946747

In [35]:
topic_info = topic_model.get_topic_info()
for idx, row in topic_info.iterrows():
    print()
    print(row["Name"])
    print()
    print(row["Representation"])
    print()
    print(row["Representative_Docs"])
    print()
    print("*"*30)


-1_storms_tripod_ships_jason

['storms', 'tripod', 'ships', 'jason', 'harbours', 'came', 'shallows', 'said', 'set', 'storm']

['following story also told said jason argo built foot pelion put aboard besides hecatomb bronze tripod set sail around peloponnese go delphi malea north wind caught carried away libya saw land came shallows tritonian lake could find way yet triton story goes appeared told jason give tripod promising show sailors channel send way unharmed jason triton showed channel shallows set tripod temple first prophesied declaring whole matter jason comrades namely descendant argo crew take away tripod hundred greek cities would founded shores tritonian lake hearing said libyan people country hid tripod', 'fleet say set forth sailed put land region magnesia beach city casthanaia headland sepias first ships came lay moored land others rode anchor behind beach large extent lay anchor prows projecting towards sea order eight ships deep night lay thus early dawn clear sky wind

In [24]:
topic_model.visualize_heatmap()

In [25]:
topic_model.visualize_topics()

In [26]:
# df["topics"] = [topic_model.topic_labels_[x] for x in topics]
# df["label"] = [id2label[i] for i in list(df["label"])]

In [27]:
# len(df["topics"].unique())

In [28]:
# df.head()

In [29]:
# normalized_counts = pd.crosstab(
#     df['label'],
#     df['topics'],
#     normalize='index'   # normalize per label (row-wise)
# )
#
# print(normalized_counts)

In [30]:
# import matplotlib.pyplot as plt
#
# normalized_counts.plot(
#     kind='barh',
#     stacked=True,
#     figsize=(10, 6),
#     colormap='tab20'
# )
#
# plt.xlabel("Proportion")
# plt.title("Topic Distribution per Label")
# plt.tight_layout()
# plt.show()